# 🧠 Brain Tumor Classification - V2
### Dataset: https://www.kaggle.com/datasets/sartajbhuvaji/brain-tumor-classification-mri
### ZIP: /content/drive/MyDrive/Datasets/brain-tumor-classification-mri.zip

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, zipfile, json
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras import layers, Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import classification_report, confusion_matrix

print('TF:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

zip_path = '/content/drive/MyDrive/Datasets/brain-tumor-classification-mri.zip'
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall('/content/brain')
print('Extracted!')

train_path = test_path = None
for root, dirs, files in os.walk('/content/brain'):
    for d in dirs:
        if d.lower() == 'training': train_path = os.path.join(root, d)
        if d.lower() == 'testing':  test_path  = os.path.join(root, d)

print('Train:', train_path)
print('Test: ', test_path)
print('Classes:', os.listdir(train_path))

In [ ]:
IMG_SIZE   = 224
BATCH_SIZE = 32

# preprocess_input — MobileNetV2 ka sahi preprocessing
# rescale=1./255 galat tha, isliye accuracy low thi
train_gen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=15,
    zoom_range=0.15,
    horizontal_flip=True,
    width_shift_range=0.1,
    height_shift_range=0.1,
    validation_split=0.1
)
test_gen = ImageDataGenerator(preprocessing_function=preprocess_input)

train_data = train_gen.flow_from_directory(
    train_path, target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='categorical',
    subset='training', shuffle=True
)
val_data = train_gen.flow_from_directory(
    train_path, target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='categorical',
    subset='validation', shuffle=False
)
test_data = test_gen.flow_from_directory(
    test_path, target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='categorical',
    shuffle=False
)

print('Classes:', train_data.class_indices)
print('Train:', train_data.samples, '| Val:', val_data.samples, '| Test:', test_data.samples)

In [ ]:
num_classes = len(train_data.class_indices)

base = MobileNetV2(weights='imagenet', include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
base.trainable = False

inp = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x   = base(inp, training=False)
x   = layers.GlobalAveragePooling2D()(x)
x   = layers.Dropout(0.5)(x)
x   = layers.Dense(128, activation='relu')(x)
x   = layers.Dropout(0.3)(x)
out = layers.Dense(num_classes, activation='softmax')(x)

model = Model(inp, out)
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

In [ ]:
# Phase 1: Train top layers
callbacks = [
    EarlyStopping(patience=5, restore_best_weights=True, monitor='val_accuracy'),
    ReduceLROnPlateau(factor=0.3, patience=2, monitor='val_loss', min_lr=1e-6)
]

history1 = model.fit(
    train_data, validation_data=val_data,
    epochs=15, callbacks=callbacks
)
print(f'Phase 1 Best: {max(history1.history["val_accuracy"])*100:.2f}%')

In [ ]:
# Phase 2: Fine-tune last 30 layers
base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks2 = [
    EarlyStopping(patience=5, restore_best_weights=True, monitor='val_accuracy'),
    ReduceLROnPlateau(factor=0.3, patience=2, monitor='val_loss', min_lr=1e-7)
]

history2 = model.fit(
    train_data, validation_data=val_data,
    epochs=15, callbacks=callbacks2
)
print(f'Phase 2 Best: {max(history2.history["val_accuracy"])*100:.2f}%')

In [ ]:
# Evaluate on test set
loss, acc = model.evaluate(test_data)
print(f'\nTest Accuracy: {acc*100:.2f}%')

preds        = model.predict(test_data)
pred_classes = np.argmax(preds, axis=1)
true_classes = test_data.classes
class_names  = list(test_data.class_indices.keys())

print('\nConfusion Matrix:')
print(confusion_matrix(true_classes, pred_classes))
print('\nClassification Report:')
print(classification_report(true_classes, pred_classes, target_names=class_names))

In [ ]:
save_dir = '/content/drive/MyDrive/ml_models'
os.makedirs(save_dir, exist_ok=True)

model.save(f'{save_dir}/brain_model.h5')
with open(f'{save_dir}/brain_classes.json', 'w') as f:
    json.dump(train_data.class_indices, f)

print('✅ brain_model.h5 saved!')
print('✅ brain_classes.json saved!')
print('Classes:', train_data.class_indices)